# 17 — Higher-Order Transition Memory + Shuffle Baseline

**prime-numbers-lab / Notebook 17**

This notebook extends Notebook 16 from a first-order transition-operator view to a higher-order memory test.

Core question:

> Is the residue-transition structure explained by a first-order Markov operator, or does measurable memory persist beyond one step?

The notebook keeps the locked template pattern:

- standalone imports
- reproducible prime generation
- deterministic outputs
- saved figures
- saved CSV summaries
- local output directory
- output zip creation
- optional Colab download cell

Notebook 17 tests:

1. first-order transition matrix `P`
2. empirical two-step transition matrix `P2_empirical`
3. Markov-predicted two-step matrix `P2_markov = P @ P`
4. residual heatmaps `P2_empirical - P2_markov`
5. lagged mutual information
6. shuffle baseline comparisons
7. conditional two-step chains
8. top higher-order residual transitions
9. notebook-level interpretation summary

In [ ]:
# Notebook 17 — locked template setup

import os
import math
import zipfile
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (11, 7)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 12

NOTEBOOK_ID = "17"
NOTEBOOK_NAME = "higher_order_transition_memory_shuffle_baseline"
OUTPUT_DIR = Path(f"{NOTEBOOK_ID}_{NOTEBOOK_NAME}_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 9423
rng = np.random.default_rng(RANDOM_SEED)

print("Output directory:", OUTPUT_DIR.resolve())

## 1. Prime generation

The notebook is self-contained. It generates primes up to `N_MAX` with a sieve, then analyzes transitions between eligible odd-prime residues modulo 30.

Residues coprime to 30:

\[
R = \{1,7,11,13,17,19,23,29\}
\]

For primes greater than 5, every prime lies in one of these residue classes modulo 30.

In [ ]:
# Prime generation

N_MAX = 2_000_000

def sieve_primes(n: int) -> np.ndarray:
    if n < 2:
        return np.array([], dtype=np.int64)
    sieve = np.ones(n + 1, dtype=bool)
    sieve[:2] = False
    sieve[4::2] = False
    limit = int(math.isqrt(n))
    for p in range(3, limit + 1, 2):
        if sieve[p]:
            sieve[p*p::2*p] = False
    return np.flatnonzero(sieve).astype(np.int64)

primes_all = sieve_primes(N_MAX)
primes = primes_all[primes_all > 5]
gaps = np.diff(primes)
x_left = primes[:-1]
x_right = primes[1:]

residues = np.array([1, 7, 11, 13, 17, 19, 23, 29], dtype=int)
residue_to_idx = {r: i for i, r in enumerate(residues)}
idx_to_residue = {i: r for r, i in residue_to_idx.items()}

prime_residues = np.array([residue_to_idx[int(p % 30)] for p in primes], dtype=int)

summary = {
    "N_MAX": N_MAX,
    "prime_count_all": int(len(primes_all)),
    "prime_count_gt5": int(len(primes)),
    "gap_count": int(len(gaps)),
    "residue_count": int(len(residues)),
}

pd.DataFrame([summary]).to_csv(OUTPUT_DIR / "17_dataset_summary.csv", index=False)
summary

## 2. Utility functions

We define standard transition and information-theoretic helpers.

The main comparison is:

\[
P_2^{emp}(j \mid i) - (P^2)(j \mid i)
\]

where:

- \(P\) is the empirical one-step transition operator.
- \(P_2^{emp}\) is the empirical two-step operator.
- \(P^2\) is the two-step operator predicted by first-order Markov dynamics.

In [ ]:
# Utility functions

def normalize_rows(counts: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    row_sums = counts.sum(axis=1, keepdims=True)
    return np.divide(counts, row_sums + eps)

def transition_counts(seq: np.ndarray, lag: int = 1, k: int = 8) -> np.ndarray:
    counts = np.zeros((k, k), dtype=float)
    for a, b in zip(seq[:-lag], seq[lag:]):
        counts[a, b] += 1
    return counts

def transition_matrix(seq: np.ndarray, lag: int = 1, k: int = 8) -> np.ndarray:
    return normalize_rows(transition_counts(seq, lag=lag, k=k))

def stationary_distribution(P: np.ndarray, steps: int = 10_000) -> np.ndarray:
    k = P.shape[0]
    pi = np.ones(k) / k
    for _ in range(steps):
        pi_next = pi @ P
        if np.max(np.abs(pi_next - pi)) < 1e-14:
            break
        pi = pi_next
    return pi / pi.sum()

def entropy(p: np.ndarray, eps: float = 1e-12) -> float:
    p = np.asarray(p, dtype=float)
    p = p[p > eps]
    return float(-(p * np.log2(p)).sum())

def row_entropy(P: np.ndarray) -> np.ndarray:
    return np.array([entropy(row) for row in P])

def mutual_information_lag(seq: np.ndarray, lag: int, k: int = 8, eps: float = 1e-12) -> float:
    counts = transition_counts(seq, lag=lag, k=k)
    joint = counts / (counts.sum() + eps)
    px = joint.sum(axis=1, keepdims=True)
    py = joint.sum(axis=0, keepdims=True)
    ratio = np.divide(joint, px @ py + eps)
    mask = joint > eps
    return float((joint[mask] * np.log2(ratio[mask])).sum())

def conditional_mutual_information_order2(seq: np.ndarray, k: int = 8, eps: float = 1e-12) -> float:
    # I(X_t ; X_{t+2} | X_{t+1})
    # Sum_b p(b) I(X_t ; X_{t+2} | X_{t+1}=b)
    triples = np.zeros((k, k, k), dtype=float)
    for a, b, c in zip(seq[:-2], seq[1:-1], seq[2:]):
        triples[a, b, c] += 1

    total = triples.sum() + eps
    cmi = 0.0
    for b in range(k):
        tab = triples[:, b, :]
        pb = tab.sum() / total
        if pb <= eps:
            continue
        joint = tab / (tab.sum() + eps)
        px = joint.sum(axis=1, keepdims=True)
        py = joint.sum(axis=0, keepdims=True)
        ratio = np.divide(joint, px @ py + eps)
        mask = joint > eps
        cmi += pb * float((joint[mask] * np.log2(ratio[mask])).sum())
    return float(cmi)

def l1_matrix(A: np.ndarray) -> float:
    return float(np.abs(A).sum())

def l2_matrix(A: np.ndarray) -> float:
    return float(np.sqrt((A * A).sum()))

def savefig(name: str):
    path = OUTPUT_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    print("saved:", path)

def label_residue_ticks(ax):
    ax.set_xticks(range(len(residues)))
    ax.set_xticklabels(residues)
    ax.set_yticks(range(len(residues)))
    ax.set_yticklabels(residues)

## 3. First-order and two-step transition operators

This section computes:

\[
P, \qquad P_2^{emp}, \qquad P^2, \qquad \Delta_2 = P_2^{emp} - P^2
\]

A small \(\Delta_2\) indicates first-order Markov sufficiency. A structured nonzero \(\Delta_2\) indicates higher-order memory.

In [ ]:
# First-order and two-step transition operators

P = transition_matrix(prime_residues, lag=1, k=len(residues))
P2_empirical = transition_matrix(prime_residues, lag=2, k=len(residues))
P2_markov = P @ P
P2_delta = P2_empirical - P2_markov

pi_stationary = stationary_distribution(P)
row_H = row_entropy(P)
row_H2 = row_entropy(P2_empirical)

operator_metrics = {
    "transition_l1_delta_P2_empirical_minus_markov": l1_matrix(P2_delta),
    "transition_l2_delta_P2_empirical_minus_markov": l2_matrix(P2_delta),
    "mean_row_entropy_P": float(row_H.mean()),
    "mean_row_entropy_P2_empirical": float(row_H2.mean()),
    "stationary_entropy": entropy(pi_stationary),
    "stationary_min": float(pi_stationary.min()),
    "stationary_max": float(pi_stationary.max()),
}

pd.DataFrame([operator_metrics]).to_csv(OUTPUT_DIR / "17_operator_metrics.csv", index=False)
pd.DataFrame(P, index=residues, columns=residues).to_csv(OUTPUT_DIR / "17_transition_operator_P.csv")
pd.DataFrame(P2_empirical, index=residues, columns=residues).to_csv(OUTPUT_DIR / "17_empirical_two_step_operator_P2.csv")
pd.DataFrame(P2_markov, index=residues, columns=residues).to_csv(OUTPUT_DIR / "17_markov_predicted_two_step_operator_P2.csv")
pd.DataFrame(P2_delta, index=residues, columns=residues).to_csv(OUTPUT_DIR / "17_two_step_delta_operator.csv")

operator_metrics

In [ ]:
# Figure 1 — first-order transition operator heatmap

plt.figure()
plt.imshow(P, aspect="auto")
plt.colorbar(label="P(r_{n+1} | r_n)")
plt.title("First-order transition operator P")
plt.xlabel("next residue mod30")
plt.ylabel("current residue mod30")
label_residue_ticks(plt.gca())
savefig("17_transition_operator_P_heatmap.png")

In [ ]:
# Figure 2 — empirical two-step operator heatmap

plt.figure()
plt.imshow(P2_empirical, aspect="auto")
plt.colorbar(label="empirical P(r_{n+2} | r_n)")
plt.title("Empirical two-step transition operator")
plt.xlabel("two-step residue mod30")
plt.ylabel("current residue mod30")
label_residue_ticks(plt.gca())
savefig("17_empirical_two_step_operator_heatmap.png")

In [ ]:
# Figure 3 — Markov-predicted two-step operator heatmap

plt.figure()
plt.imshow(P2_markov, aspect="auto")
plt.colorbar(label="Markov P²")
plt.title("Markov-predicted two-step operator P²")
plt.xlabel("two-step residue mod30")
plt.ylabel("current residue mod30")
label_residue_ticks(plt.gca())
savefig("17_markov_predicted_two_step_operator_heatmap.png")

In [ ]:
# Figure 4 — higher-order residual: empirical P2 minus Markov P2

plt.figure()
plt.imshow(P2_delta, aspect="auto")
plt.colorbar(label="P2 empirical - P²")
plt.title("Two-step residual operator: empirical P₂ - P²")
plt.xlabel("two-step residue mod30")
plt.ylabel("current residue mod30")
label_residue_ticks(plt.gca())
savefig("17_two_step_operator_delta_heatmap.png")

## 4. Top higher-order residual transitions

The table below ranks the residue pairs where empirical two-step behavior differs most from the first-order Markov prediction.

In [ ]:
# Top two-step residual transitions

records = []
for i, r0 in enumerate(residues):
    for j, r2 in enumerate(residues):
        records.append({
            "r_n_mod30": int(r0),
            "r_n_plus_2_mod30": int(r2),
            "P2_empirical": float(P2_empirical[i, j]),
            "P2_markov": float(P2_markov[i, j]),
            "delta": float(P2_delta[i, j]),
            "abs_delta": float(abs(P2_delta[i, j])),
        })

top_two_step = pd.DataFrame(records).sort_values("abs_delta", ascending=False)
top_two_step.to_csv(OUTPUT_DIR / "17_top_two_step_residual_transitions.csv", index=False)
top_two_step.head(20)

In [ ]:
# Figure 5 — top two-step residual bars

topN = top_two_step.head(20).copy()
labels = [f"{a}->{b}" for a, b in zip(topN["r_n_mod30"], topN["r_n_plus_2_mod30"])]

plt.figure(figsize=(13, 7))
plt.bar(labels, topN["delta"])
plt.axhline(0, linestyle="--")
plt.title("Top two-step residual transitions")
plt.xlabel("r_n -> r_{n+2} mod30")
plt.ylabel("P2 empirical - P²")
plt.xticks(rotation=45, ha="right")
savefig("17_top_two_step_residual_transitions.png")

## 5. Lagged mutual information

We compute:

\[
I(r_n; r_{n+k})
\]

for several lags. If structure decays quickly, the curve drops toward the shuffle baseline.

In [ ]:
# Lagged mutual information and shuffle baseline

MAX_LAG = 12
N_SHUFFLES = 24

mi_real = []
mi_shuffle_mean = []
mi_shuffle_std = []

for lag in range(1, MAX_LAG + 1):
    real_val = mutual_information_lag(prime_residues, lag=lag, k=len(residues))
    sh_vals = []
    for _ in range(N_SHUFFLES):
        shuffled = rng.permutation(prime_residues)
        sh_vals.append(mutual_information_lag(shuffled, lag=lag, k=len(residues)))
    mi_real.append(real_val)
    mi_shuffle_mean.append(float(np.mean(sh_vals)))
    mi_shuffle_std.append(float(np.std(sh_vals)))

mi_df = pd.DataFrame({
    "lag": np.arange(1, MAX_LAG + 1),
    "mi_real_bits": mi_real,
    "mi_shuffle_mean_bits": mi_shuffle_mean,
    "mi_shuffle_std_bits": mi_shuffle_std,
    "mi_excess_bits": np.array(mi_real) - np.array(mi_shuffle_mean),
})

cmi_order2 = conditional_mutual_information_order2(prime_residues, k=len(residues))

mi_summary = {
    "conditional_mutual_information_I_rn_rn2_given_rn1_bits": cmi_order2,
    "lag1_excess_bits": float(mi_df.loc[mi_df["lag"] == 1, "mi_excess_bits"].iloc[0]),
    "lag2_excess_bits": float(mi_df.loc[mi_df["lag"] == 2, "mi_excess_bits"].iloc[0]),
    "mean_excess_lag1_to_12_bits": float(mi_df["mi_excess_bits"].mean()),
}

mi_df.to_csv(OUTPUT_DIR / "17_lagged_mutual_information.csv", index=False)
pd.DataFrame([mi_summary]).to_csv(OUTPUT_DIR / "17_mutual_information_summary.csv", index=False)

mi_summary

In [ ]:
# Figure 6 — mutual information vs lag

plt.figure()
plt.plot(mi_df["lag"], mi_df["mi_real_bits"], marker="o", label="real")
plt.plot(mi_df["lag"], mi_df["mi_shuffle_mean_bits"], marker="o", linestyle="--", label="shuffle mean")
plt.fill_between(
    mi_df["lag"],
    mi_df["mi_shuffle_mean_bits"] - mi_df["mi_shuffle_std_bits"],
    mi_df["mi_shuffle_mean_bits"] + mi_df["mi_shuffle_std_bits"],
    alpha=0.2,
    label="shuffle ±1 std",
)
plt.title("Lagged mutual information")
plt.xlabel("lag k")
plt.ylabel("I(r_n ; r_{n+k}) [bits]")
plt.legend()
savefig("17_mutual_information_vs_lag.png")

In [ ]:
# Figure 7 — excess mutual information

plt.figure()
plt.plot(mi_df["lag"], mi_df["mi_excess_bits"], marker="o")
plt.axhline(0, linestyle="--")
plt.title("Excess mutual information above shuffle baseline")
plt.xlabel("lag k")
plt.ylabel("excess MI [bits]")
savefig("17_excess_mutual_information_vs_lag.png")

## 6. Conditional two-step chains

Now we condition on the intermediate residue:

\[
P(r_{n+2} \mid r_n, r_{n+1})
\]

This identifies whether specific chains carry residual structure that a one-step operator erases.

In [ ]:
# Conditional two-step chain table

k = len(residues)
triple_counts = np.zeros((k, k, k), dtype=float)

for a, b, c in zip(prime_residues[:-2], prime_residues[1:-1], prime_residues[2:]):
    triple_counts[a, b, c] += 1

chain_records = []
for a in range(k):
    for b in range(k):
        total_ab = triple_counts[a, b, :].sum()
        if total_ab == 0:
            continue
        empirical_cond = triple_counts[a, b, :] / total_ab
        markov_cond = P[b, :]
        delta = empirical_cond - markov_cond
        for c in range(k):
            chain_records.append({
                "r_n_mod30": int(residues[a]),
                "r_n_plus_1_mod30": int(residues[b]),
                "r_n_plus_2_mod30": int(residues[c]),
                "count_chain_prefix": int(total_ab),
                "P_empirical_given_chain": float(empirical_cond[c]),
                "P_markov_given_middle": float(markov_cond[c]),
                "delta": float(delta[c]),
                "abs_delta": float(abs(delta[c])),
            })

chain_df = pd.DataFrame(chain_records).sort_values("abs_delta", ascending=False)
chain_df.to_csv(OUTPUT_DIR / "17_conditional_two_step_chain_residuals.csv", index=False)
chain_df.head(25)

In [ ]:
# Figure 8 — top conditional two-step chain residuals

top_chain = chain_df.head(25).copy()
labels = [
    f"{a}->{b}->{c}"
    for a, b, c in zip(
        top_chain["r_n_mod30"],
        top_chain["r_n_plus_1_mod30"],
        top_chain["r_n_plus_2_mod30"],
    )
]

plt.figure(figsize=(14, 7))
plt.bar(labels, top_chain["delta"])
plt.axhline(0, linestyle="--")
plt.title("Top conditional two-step chain residuals")
plt.xlabel("r_n -> r_{n+1} -> r_{n+2} mod30")
plt.ylabel("empirical conditional - Markov conditional")
plt.xticks(rotation=55, ha="right")
savefig("17_top_conditional_chain_residuals.png")

## 7. Real vs shuffled transition residuals

This is the notebook's key falsification check.

We compare real two-step residual magnitude against shuffled sequences. If the real value is consistently above shuffled values, the higher-order structure is not just a finite-sample artifact.

In [ ]:
# Shuffle baseline for P2 residual operator

shuffle_records = []

real_l1 = l1_matrix(P2_delta)
real_l2 = l2_matrix(P2_delta)

for s in range(N_SHUFFLES):
    shuffled = rng.permutation(prime_residues)
    P_sh = transition_matrix(shuffled, lag=1, k=k)
    P2_sh_emp = transition_matrix(shuffled, lag=2, k=k)
    P2_sh_markov = P_sh @ P_sh
    delta_sh = P2_sh_emp - P2_sh_markov

    shuffle_records.append({
        "shuffle_id": s,
        "l1_delta": l1_matrix(delta_sh),
        "l2_delta": l2_matrix(delta_sh),
        "mean_entropy_P": float(row_entropy(P_sh).mean()),
        "mean_entropy_P2_empirical": float(row_entropy(P2_sh_emp).mean()),
    })

shuffle_df = pd.DataFrame(shuffle_records)
shuffle_df.to_csv(OUTPUT_DIR / "17_shuffle_two_step_residual_baseline.csv", index=False)

shuffle_summary = {
    "real_l1_delta": real_l1,
    "shuffle_l1_mean": float(shuffle_df["l1_delta"].mean()),
    "shuffle_l1_std": float(shuffle_df["l1_delta"].std()),
    "real_l2_delta": real_l2,
    "shuffle_l2_mean": float(shuffle_df["l2_delta"].mean()),
    "shuffle_l2_std": float(shuffle_df["l2_delta"].std()),
    "real_l1_zscore_vs_shuffle": float((real_l1 - shuffle_df["l1_delta"].mean()) / (shuffle_df["l1_delta"].std() + 1e-12)),
    "real_l2_zscore_vs_shuffle": float((real_l2 - shuffle_df["l2_delta"].mean()) / (shuffle_df["l2_delta"].std() + 1e-12)),
}
pd.DataFrame([shuffle_summary]).to_csv(OUTPUT_DIR / "17_shuffle_residual_summary.csv", index=False)
shuffle_summary

In [ ]:
# Figure 9 — shuffle comparison, L1

plt.figure()
plt.hist(shuffle_df["l1_delta"], bins=12, alpha=0.8, label="shuffle")
plt.axvline(real_l1, linestyle="--", linewidth=2, label="real")
plt.title("Two-step residual L1: real vs shuffled")
plt.xlabel("L1(P2 empirical - P²)")
plt.ylabel("frequency")
plt.legend()
savefig("17_real_vs_shuffle_two_step_l1.png")

In [ ]:
# Figure 10 — shuffle comparison, L2

plt.figure()
plt.hist(shuffle_df["l2_delta"], bins=12, alpha=0.8, label="shuffle")
plt.axvline(real_l2, linestyle="--", linewidth=2, label="real")
plt.title("Two-step residual L2: real vs shuffled")
plt.xlabel("L2(P2 empirical - P²)")
plt.ylabel("frequency")
plt.legend()
savefig("17_real_vs_shuffle_two_step_l2.png")

## 8. Windowed higher-order residual drift

We compute higher-order residual strength by scale window to test whether two-step memory stabilizes, grows, or disappears with larger \(x\).

In [ ]:
# Windowed higher-order residual drift

N_WINDOWS = 18
valid_len = len(prime_residues)
window_edges = np.linspace(0, valid_len - 2, N_WINDOWS + 1, dtype=int)

window_records = []

for w in range(N_WINDOWS):
    a, b = window_edges[w], window_edges[w + 1]
    seq_w = prime_residues[a:b + 2]
    x_mid = float(np.median(primes[a:b + 2]))

    if len(seq_w) < 50:
        continue

    P_w = transition_matrix(seq_w, lag=1, k=k)
    P2_w_emp = transition_matrix(seq_w, lag=2, k=k)
    P2_w_markov = P_w @ P_w
    delta_w = P2_w_emp - P2_w_markov

    window_records.append({
        "window_index": w,
        "x_mid": x_mid,
        "sample_count": int(len(seq_w)),
        "l1_delta": l1_matrix(delta_w),
        "l2_delta": l2_matrix(delta_w),
        "mean_row_entropy_P": float(row_entropy(P_w).mean()),
        "mean_row_entropy_P2": float(row_entropy(P2_w_emp).mean()),
        "mi_lag1_bits": mutual_information_lag(seq_w, lag=1, k=k),
        "mi_lag2_bits": mutual_information_lag(seq_w, lag=2, k=k),
        "conditional_mi_order2_bits": conditional_mutual_information_order2(seq_w, k=k),
    })

window_df = pd.DataFrame(window_records)
window_df.to_csv(OUTPUT_DIR / "17_windowed_higher_order_residual_metrics.csv", index=False)
window_df

In [ ]:
# Figure 11 — windowed residual norms

plt.figure()
plt.plot(window_df["x_mid"], window_df["l1_delta"], marker="o", label="L1 residual")
plt.plot(window_df["x_mid"], window_df["l2_delta"], marker="o", label="L2 residual")
plt.xscale("log")
plt.title("Windowed higher-order residual norm")
plt.xlabel("window midpoint x")
plt.ylabel("residual norm")
plt.legend()
savefig("17_windowed_two_step_residual_norms.png")

In [ ]:
# Figure 12 — windowed lag MI comparison

plt.figure()
plt.plot(window_df["x_mid"], window_df["mi_lag1_bits"], marker="o", label="MI lag 1")
plt.plot(window_df["x_mid"], window_df["mi_lag2_bits"], marker="o", label="MI lag 2")
plt.plot(window_df["x_mid"], window_df["conditional_mi_order2_bits"], marker="o", label="CMI order 2")
plt.xscale("log")
plt.title("Windowed information structure")
plt.xlabel("window midpoint x")
plt.ylabel("bits")
plt.legend()
savefig("17_windowed_information_structure.png")

## 9. Transition-lift comparison

We compare observed transition probability against the independent stationary baseline:

\[
\text{lift}(i,j) = \frac{P(j \mid i)}{\pi(j)}
\]

and extend the idea to empirical two-step lift.

In [ ]:
# Transition lift and two-step lift

eps = 1e-12
lift_P = P / (pi_stationary.reshape(1, -1) + eps)
pi2_stationary = pi_stationary  # same residue state space
lift_P2 = P2_empirical / (pi2_stationary.reshape(1, -1) + eps)
lift_delta = lift_P2 - lift_P

pd.DataFrame(lift_P, index=residues, columns=residues).to_csv(OUTPUT_DIR / "17_lift_first_order.csv")
pd.DataFrame(lift_P2, index=residues, columns=residues).to_csv(OUTPUT_DIR / "17_lift_two_step.csv")
pd.DataFrame(lift_delta, index=residues, columns=residues).to_csv(OUTPUT_DIR / "17_lift_two_step_minus_first_order.csv")

lift_metrics = {
    "first_order_lift_min": float(np.min(lift_P)),
    "first_order_lift_max": float(np.max(lift_P)),
    "two_step_lift_min": float(np.min(lift_P2)),
    "two_step_lift_max": float(np.max(lift_P2)),
    "lift_delta_l1": l1_matrix(lift_delta),
}
pd.DataFrame([lift_metrics]).to_csv(OUTPUT_DIR / "17_lift_metrics.csv", index=False)
lift_metrics

In [ ]:
# Figure 13 — first-order lift heatmap

plt.figure()
plt.imshow(lift_P, aspect="auto")
plt.colorbar(label="lift P(j|i)/pi(j)")
plt.title("First-order transition lift")
plt.xlabel("next residue mod30")
plt.ylabel("current residue mod30")
label_residue_ticks(plt.gca())
savefig("17_first_order_transition_lift_heatmap.png")

In [ ]:
# Figure 14 — two-step lift heatmap

plt.figure()
plt.imshow(lift_P2, aspect="auto")
plt.colorbar(label="two-step lift")
plt.title("Two-step transition lift")
plt.xlabel("two-step residue mod30")
plt.ylabel("current residue mod30")
label_residue_ticks(plt.gca())
savefig("17_two_step_transition_lift_heatmap.png")

In [ ]:
# Figure 15 — lift difference heatmap

plt.figure()
plt.imshow(lift_delta, aspect="auto")
plt.colorbar(label="two-step lift - first-order lift")
plt.title("Two-step lift minus first-order lift")
plt.xlabel("target residue mod30")
plt.ylabel("source residue mod30")
label_residue_ticks(plt.gca())
savefig("17_lift_delta_heatmap.png")

## 10. Interpretation numbers

Notebook 17 should produce concrete interpretation values so it is not just a figure notebook.

In [ ]:
# Interpretation summary

interpretation = {
    **summary,
    **operator_metrics,
    **mi_summary,
    **shuffle_summary,
    **lift_metrics,
}

interpretation_df = pd.DataFrame([interpretation])
interpretation_df.to_csv(OUTPUT_DIR / "17_interpretation_summary.csv", index=False)

print("Notebook 17 interpretation summary")
for key, value in interpretation.items():
    if isinstance(value, float):
        print(f"{key}: {value:.8g}")
    else:
        print(f"{key}: {value}")

## 11. Notebook 17 conclusion

Notebook 17 upgrades residue-transition analysis from first-order operator structure to higher-order memory testing.

A successful result is not “huge memory.” The useful signal is:

- empirical two-step transitions are measurable,
- Markov-predicted two-step transitions are computable,
- their residual has ranked structure,
- shuffle baselines quantify whether residual is above finite-sample noise,
- lagged mutual information shows whether residue dependence persists,
- conditional chain residuals identify specific memory paths.

This notebook prepares the repo for Notebook 18:

> **operator compression / spectral representation / low-rank transition structure**

## 12. Save outputs manifest and optional download cell

The output zip is created inside the notebook. In Colab, uncomment the final two lines to download it.

In [ ]:
# Output manifest and zip creation

manifest_rows = []
for path in sorted(OUTPUT_DIR.glob("*")):
    manifest_rows.append({
        "filename": path.name,
        "relative_path": str(path),
        "size_bytes": path.stat().st_size,
    })

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(OUTPUT_DIR / "17_outputs_manifest.csv", index=False)

ZIP_NAME = "17_higher_order_transition_memory_shuffle_baseline_outputs.zip"

with zipfile.ZipFile(ZIP_NAME, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTPUT_DIR.glob("*")):
        zf.write(path, arcname=f"{OUTPUT_DIR.name}/{path.name}")

print("Created:", ZIP_NAME)
print("Files included:", len(manifest_df))
manifest_df

In [ ]:
# Optional: download outputs bundle in Google Colab
# from google.colab import files
# files.download("17_higher_order_transition_memory_shuffle_baseline_outputs.zip")